In [2]:
import pandas as pd
import torch
import matplotlib.pyplot as plt
import numpy as np

from exp.exp_forecasting import Exp_Forecasting
from exp.exp_classification import Exp_Classification
from utils.tools import (
    set_seed,
    print_formatted_dict,
)
from dataset_loader.dataset_loader import update_args_from_dataset
from main import trainable, get_args_from_parser, update_args

In [2]:
train_dataset = torch.load("./dataset/classification/WISDM/train.pt")
val_dataset = torch.load("./dataset/classification/WISDM/val.pt")
test_dataset = torch.load("./dataset/classification/WISDM/test.pt")

In [3]:
labels = train_dataset.get("labels").numpy()

unique, counts = np.unique(labels, return_counts=True)

In [4]:
print(dict(zip(unique, counts)))

{0: 1027, 2: 152, 3: 122, 5: 203}


# Train model

In [5]:
task_name = "classification"

data_name = "WISDM"  # (256, 3)

num_workers = 4  # It doesn't really matter since we don't use it in dataloader
batch_size = 16

train_together = False  # default
get_i = "cls"
encoder_arch = "transformer_encoder"  # default
data_aug = "none"
disable_stop_gradient = False
disable_predictive_loss = False  # default
disable_contrastive_loss = False  # default
pretrain_data_percent = 100  # default
linear_eval_data_percent = 100  # default
disable_freeze_encoder = False  # default
set_seed(seed=2023)

# Setup args
args = get_args_from_parser()
args.overwrite_args = True

# Setup fixed params
fixed_params = {
    "task_name": task_name,
    "data_name": data_name,
    "num_workers": num_workers,
    "batch_size": batch_size,
    "train_together": train_together,
    "get_i": get_i,
    "encoder_arch": encoder_arch,
    "data_aug": data_aug,
    "disable_stop_gradient": disable_stop_gradient,
    "disable_predictive_loss": disable_predictive_loss,
    "disable_contrastive_loss": disable_contrastive_loss,
    "pretrain_data_percent": pretrain_data_percent,
    "linear_eval_data_percent": linear_eval_data_percent,
    "disable_freeze_encoder": disable_freeze_encoder,
}

# Setup tunable params
# TODO: copy `config` from `exp_settings_and_results` (be careful with the boolean values)
tunable_params = {
    "pretrain_optim": "AdamW",
    "pretrain_learning_rate": 0.0001407743941654141,
    "pretrain_lradj": "constant",
    "pretrain_weight_decay": 0.00008709261322555189,
    "pretrain_epochs": 10, # 30 - best
    "contrastive_weight": 0.1,
    "linear_eval_optim": "AdamW",
    "linear_eval_learning_rate": 0.0014077439416541409,
    "linear_eval_lradj": "constant",
    "linear_eval_weight_decay": 0.00017514842046704846,
    "linear_eval_epochs": 7, # 30 - best
    "patience": 5,
    # "pos_embed_type": "learnable",
    # "token_embed_type": "conv",
    # "token_embed_kernel_size": 3,
    # "dropout": 0.2,
    # "base_d_model": 64,
    # "n_layers": 1,
    # "n_heads": 2,
    # "patch_len": 16,
    # "stride": 16,
    # "enable_channel_independence": True,
    # "seq_len": 128,
    "pos_embed_type": "learnable",
    "token_embed_type": "conv",
    "token_embed_kernel_size": 5,
    "dropout": 0.2,
    "base_d_model": 128,
    "n_layers": 3,
    "n_heads": 2,
    "patch_len": 4,
    "stride": 4,
    "enable_channel_independence": False,
    "seq_len": 168
}

In [ ]:
return_metrics = trainable(tunable_params, fixed_params, args)
print_formatted_dict(return_metrics)

In [6]:
# 1. Setup and Update Args
args = update_args(args, fixed_params, tunable_params)

# 2. Initialize the Experiment
exp = Exp_Classification(args)

# 3. Run Training (this trains both encoder and linear head)
if args.train_together:
    train_metrics = exp.train_together(use_tqdm=True)
else:
    train_metrics = exp.train(use_tqdm=True)

### [Fixed] Set task_name to classification
### [Fixed] Set data_name to WISDM
### [Fixed] Set num_workers to 4
### [Fixed] Set batch_size to 16
### [Fixed] Set train_together to False
### [Fixed] Set get_i to cls
### [Fixed] Set encoder_arch to transformer_encoder
### [Fixed] Set data_aug to none
### [Fixed] Set disable_stop_gradient to False
### [Fixed] Set disable_predictive_loss to False
### [Fixed] Set disable_contrastive_loss to False
### [Fixed] Set pretrain_data_percent to 100
### [Fixed] Set linear_eval_data_percent to 100
### [Fixed] Set disable_freeze_encoder to False
### [Tunable] Set pretrain_optim to AdamW
### [Tunable] Set pretrain_learning_rate to 0.0001407743941654141
### [Tunable] Set pretrain_lradj to constant
### [Tunable] Set pretrain_weight_decay to 8.709261322555189e-05
### [Tunable] Set pretrain_epochs to 10
### [Tunable] Set contrastive_weight to 0.1
### [Tunable] Set linear_eval_optim to AdamW
### [Tunable] Set linear_eval_learning_rate to 0.001407743941654140

Class Distribution Across Datasets:

┏━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Class ┃         Train ┃   Validation ┃         Test ┃
┡━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│     0 │ 1027 (68.28%) │ 263 (69.21%) │ 339 (70.92%) │
│     2 │  152 (10.11%) │   31 (8.16%) │   38 (7.95%) │
│     3 │   122 (8.11%) │   35 (9.21%) │   23 (4.81%) │
│     5 │  203 (13.50%) │  51 (13.42%) │  78 (16.32%) │
└───────┴───────────────┴──────────────┴──────────────┘

----------------------------------------
### WISDM ###
N: 2362 (train: 1504, val: 380, test: 478)
C: 3
K: 4
T: 256


Class Distribution Across Datasets:

...

100%|██████████| 24/24 [00:19<00:00,  1.23it/s]


>>>>> Calculate testing metrics >>>>>


100%|██████████| 30/30 [00:24<00:00,  1.22it/s]


┏━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━┓
┃ Epoch ┃ Valid LOSS ┃ Valid ACC ┃ Valid MF1 ┃ Valid KAPPA ┃ Test LOSS ┃ Test ACC ┃ Test MF1 ┃ Test KAPPA ┃
┡━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━┩
│ 1     │ 0.028      │ 0.824     │ 0.645     │ 0.594       │ 0.031     │ 0.814    │ 0.617    │ 0.541      │
└───────┴────────────┴───────────┴───────────┴─────────────┴───────────┴──────────┴──────────┴────────────┘

Early stopping
Updating learning rate to 0.0001407743941654141
------------------------------------------------------------------
best_pretrain_loss: 0.104
predictive_loss: 0.186
contrastive_loss: -0.815
best_test_acc: 0.816
best_test_mf1: 0.635
best_test_kappa: 0.546
best_pretrain_epoch: 10
best_best_test_acc_epoch: 7


In [12]:
# 4. GET THE DATA LOADERS
# We need the loaders to pass into your new analysis function
train_loader, val_loader, test_loader = exp._get_data()

# 5. CALL THE ANALYSIS FUNCTION
print(">>>>> Running Detailed Analysis for OOD/Confusion Matrix >>>>>")
analysis_results = exp.get_detailed_analysis_dict(test_loader)

# Now you have everything!
print(f"Latents shape: {analysis_results['latents'].shape}")
print(f"Logits shape: {analysis_results['logits'].shape}")

# Example: Accessing targets and preds for a custom confusion matrix
from sklearn.metrics import confusion_matrix
y_true = analysis_results['targets']
y_pred = analysis_results['preds']
print(confusion_matrix(y_true, y_pred))

----------------------------------------
### WISDM ###
N: 1729 (train: 1113, val: 275, test: 341)
C: 3
K: 2
T: 256


Class Distribution Across Datasets:

┏━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ Class ┃        Train ┃   Validation ┃         Test ┃
┡━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│     1 │ 830 (74.57%) │ 225 (81.82%) │ 258 (75.66%) │
│     4 │ 283 (25.43%) │  50 (18.18%) │  83 (24.34%) │
└───────┴──────────────┴──────────────┴──────────────┘

>>>>> Running Detailed Analysis for OOD/Confusion Matrix >>>>>
Latents shape: (341, 256)
Logits shape: (341, 6)
[[  0   0   0   0   0   0]
 [130   0   8   1   0 119]
 [  0   0   0   0   0   0]
 [  0   0   0   0   0   0]
 [ 52   0  10   7   0  14]
 [  0   0   0   0   0   0]]


In [13]:
# To Save
file_path = "WISDM_n3_h2_ood.npz"
np.savez(file_path, **analysis_results)
print(f"Saved analysis results to {file_path}")

Saved analysis results to WISDM_n3_h2_ood.npz


In [8]:
# Key,Source in TimeDRL,Purpose in OOD Detection
# latents,Output of the Encoder (i1​),"These are the ""raw"" feature vectors. In OOD detection, you use these to see if OOD samples cluster in a different area of the latent space than the training classes."
# logits,Output of the Linear Head,"The raw scores before normalization. These are essential for Energy-based scores, which are often more reliable for OOD than probabilities."
# probs,softmax(logits),"The probability distribution across all known classes. For OOD data, you expect this to be ""flatter"" (higher entropy)."
# preds,argmax(probs),"The final class assignment. For OOD data, the model is forced to choose a class, which is technically a ""false positive."""
# targets,Original Ground Truth,Used to calculate the Confusion Matrix and to identify which samples were your removed OOD classes.
# max_conf,max(probs),"The Maximum Softmax Probability (MSP). This is your primary baseline score: high for known classes, low for OOD."

so i should train the model using only the 4 class train data and validate it with a test and val dataset where also only those 4 classes exist, then after the training i should run this get_detailed_analysis_dict function with 2 cases: the first where i should only give it those 4 classes that the model knows and second only those 2 classes that it doesnt know?